<a href="https://colab.research.google.com/github/keerthis-95/Triton-practice/blob/main/Triton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 16.7 MB/s  0:00:16


#Triton

- Use torch.compile() as it makes new simple kernels and use existing kernels efficiently

- do your model -> if not fast enough use torch.compile -> if that's not fast enough improvise for torch.compile -> if that's not fast enough write custom triton kernels -> if that's not fast enough write custom CUDA kernels.

- set global variable TRITON_INTERPRET = 1 to debug triton like any cpu program (do this before importing triton).

- PS:Every cell below are altere multiple times for each variation of code (which means my practice is much more than the number of cells below)

##adding tensor-float and tensor-tensor

In [ ]:
import torch
import triton.language as tl
import triton

@triton.jit
def add_kernel(x_ptr, y_ptr, size):
  pid = tl.program_id(0) #which row/block the program handles

  if pid<size:
    x = tl.load(x_ptr + pid)
    y = x + 1.0
    tl.store(y_ptr + pid, y)

def add_one(x):
  x = x.to('cuda') #move to gpu
  y = torch.empty_like(x)

  size = x.numel()
  grid = (size,)

  add_kernel[grid](x,y,size)
  return y

x = torch.tensor([1.0, 2.0, 3.0, 4.0],device='cuda')
print(x.cpu())
y = add_one(x)
print(y.cpu()) #.cpu is used as print cannot directly access cuda tensors

#Adding two tensors
@triton.jit
def add_kernel(x_ptr, y_ptr, output, size):
  pid = tl.program_id(0)

  if pid<size:
    x = tl.load(x_ptr + pid)
    y = tl.load(y_ptr + pid)
    tl.store(output + pid, x+y)

def add(x,y):
  x = x.to('cuda')
  y = y.to('cuda')
  output = torch.empty_like(x)

  size = x.numel()
  grid = (size,)

  add_kernel[grid](x,y,output,size)
  return output

x = torch.tensor([1.0, 2.0, 3.0, 4.0], device = 'cuda')
y = torch.tensor([4.0, 3.0, 2.0, 1.0], device = 'cuda')
z = add(x,y)
print(z)

tensor([1., 2., 3., 4.])
tensor([2., 3., 4., 5.])


##dot product and ReLU

In [ ]:
#This program was initially a dot-product program , I wanted to try ReLU and made alterations.
import torch
import triton
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, size, BLOCK_SIZE:tl.constexpr):
  pid = tl.program_id(0)
  offset = pid*BLOCK_SIZE + tl.arange(0,BLOCK_SIZE)
  mask = offset<size
  x = tl.load(x_ptr + offset, mask=mask, other=0.0)
  y = tl.load(y_ptr + offset, mask=mask, other=0.0)
  tl.store(output_ptr + offset, tl.maximum(y,0.0))

def dot_product(x,y):
  assert x.numel()==y.numel()
  x = x.to('cuda')
  y = y.to('cuda')
  size = x.numel()

  output = torch.empty_like(y)
  BLOCK_SIZE = 1024
  num_blocks = (size + BLOCK_SIZE - 1)//BLOCK_SIZE
  add_kernel[(num_blocks,)](x, y, output, size, BLOCK_SIZE=BLOCK_SIZE)
  return output

x = torch.tensor([1.0, 2.0, 3.0, 4.0], device = 'cuda')
y = torch.tensor([-10.0, 20.0, -30.0, 40.0], device = 'cuda')

print('Triton relu:', dot_product(x,y))
print('Torch relu:', torch.relu(y))

Triton relu: tensor([ 0., 20.,  0., 40.], device='cuda:0')
Torch relu: tensor([ 0., 20.,  0., 40.], device='cuda:0')


In [ ]:
#element-wise linear kernel
@triton.jit
def add_kernel(x,a,b,output,size,BLOCK_SIZE:tl.constexpr):
  pid = tl.program_id(0)
  offset = pid*BLOCK_SIZE + tl.arange(0,BLOCK_SIZE)
  mask = offset<size

  x_ptr = tl.load(x + offset, mask = mask)
  tl.store(output + offset, (x_ptr*a)+b)

def add(x,a,b):
  output = torch.empty_like(x)
  size = x.numel()
  BLOCK_SIZE = 1024
  num_blocks = (size + BLOCK_SIZE - 1)//BLOCK_SIZE
  add_kernel[(num_blocks,)](x,a,b,output,size,BLOCK_SIZE=BLOCK_SIZE)
  return output

x = torch.tensor([1.0, 2.0, 3.0, 4.0],device = 'cuda')
a = 2.0
b = 1.0
print(add(x,a,b))

tensor([3., 5., 7., 9.], device='cuda:0')


##Matrix addition

In [ ]:
@triton.jit
def add_kernel(x, y, output, rows, cols):
  row = tl.program_id(0)
  col = tl.program_id(1)

  if row<rows and col<cols:
    pid = row*cols + col #converts 2D position to 1D
    x_ptr = tl.load(x + pid)
    tl.store(output+pid, x_ptr + y)

def add_2d(x,y):
  x = x.to('cuda')
  output = torch.empty_like(x)

  rows,cols = x.shape
  add_kernel[(rows,cols)](x,y,output,rows,cols)

x = torch.tensor([[1., 2.],
                  [3., 4.]], device="cuda")

y = 3.0

print(add_2d(x, y))
print(x + y)

None
tensor([[4., 5.],
        [6., 7.]], device='cuda:0')


###Summing each row

In [ ]:
#GO THROUGH LATER
@triton.jit
def add_kernel(x, output, rows, cols, BLOCK_SIZE:tl.constexpr):
  rowid = tl.program_id(0)

  if rowid<rows:
    col_offset = tl.arange(0, BLOCK_SIZE)
    mask = col_offset < cols
    offsets = rowid*cols + col_offset

    vals = tl.load(x + offsets , mask=mask, other = float('-inf'))
    row_sum = tl.max(vals, axis=0)
    tl.store(output + rowid, row_sum)

def add_row(x):
  x = x.to('cuda')
  rows,cols = x.shape
  output = torch.empty((rows,), device='cuda', dtype=x.dtype)
  BLOCK_SIZE = triton.next_power_of_2(cols)
  add_kernel[(rows,)](x, output, rows, cols, BLOCK_SIZE=BLOCK_SIZE)
  return output

x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]], device="cuda")
print(add_row(x))

tensor([3., 6.], device='cuda:0')
